# 🚀 SNOMED CT Entity Linking - Qwen2.5 Fine-tuning

**Google Colab용 학습 노트북**

이 노트북을 사용하면:
- ✅ GPU 환경에서 Qwen2.5 fine-tuning
- ✅ 자동으로 모델 다운로드 및 설정
- ✅ Google Drive에 모델 저장
- ✅ 4-6시간 후 학습 완료

---

## 📋 사전 준비

1. **런타임 → 런타임 유형 변경 → GPU (T4)** 선택
2. 학습 데이터를 Colab에 업로드 (또는 Drive에 저장)
3. 아래 셀들을 순서대로 실행

## Step 1: GPU 확인

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: 저장소 클론

In [ ]:
!git clone https://github.com/ho0n-SNUH/snomed-ct-entity-linking.git
%cd snomed-ct-entity-linking
!git checkout claude/incomplete-description-011CV1AmjEgryVZLyTuDTwE2

# 디렉토리 구조 확인
!ls -la

## Step 3: 의존성 설치

In [ ]:
# 필수 패키지 설치
!pip install -q transformers[torch]>=4.37.0
!pip install -q accelerate>=0.25.0
!pip install -q peft>=0.8.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q datasets
!pip install -q trl
!pip install -q pandas
!pip install -q scikit-learn

print("✅ 모든 패키지 설치 완료!")

## Step 4: Google Drive 마운트 (모델 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 모델 저장 디렉토리 생성
import os
MODEL_SAVE_DIR = '/content/drive/MyDrive/snomed-ct-models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"✅ 모델은 다음 경로에 저장됩니다: {MODEL_SAVE_DIR}")

## Step 5: 학습 데이터 준비

**방법 1**: Colab Files 패널에서 직접 업로드

**방법 2**: Google Drive에서 복사

In [ ]:
# 옵션 A: 로컬 파일 업로드
from google.colab import files
print("학습 데이터 (train_notes.csv, train_annotations.csv)를 업로드하세요:")
uploaded = files.upload()

# data 디렉토리에 저장
import shutil
os.makedirs('data', exist_ok=True)
for filename in uploaded.keys():
    shutil.move(filename, f'data/{filename}')

print("\n✅ 업로드된 파일:")
!ls -lh data/

In [ ]:
# 옵션 B: Google Drive에서 복사
# Drive에 데이터가 있다면 이 셀 사용

# 예: /content/drive/MyDrive/snomed-data/ 에 데이터가 있다면
# !cp /content/drive/MyDrive/snomed-data/*.csv data/

# 데이터 확인
!ls -lh data/

## Step 6: Fine-tuning 설정

In [ ]:
# 설정 변수
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # 또는 "Qwen/Qwen2.5-14B-Instruct"
TASK = "entity_recognition"  # 또는 "classification"
CHUNK_SIZE = 100  # 100 또는 500

import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{MODEL_SAVE_DIR}/qwen-{TASK}-{CHUNK_SIZE}tok-{timestamp}"

print(f"Base Model: {BASE_MODEL}")
print(f"Task: {TASK}")
print(f"Output: {OUTPUT_DIR}")

## Step 7-A: Entity Recognition Fine-tuning

In [ ]:
# Entity Recognition 학습 (약 4-6시간)

# 먼저 기존 스크립트를 Qwen으로 수정
# (실제 환경에서는 Finetuning-Entity-Recognition.py를 수정해야 함)

# 간단한 LoRA fine-tuning 코드
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch

print("Loading model and tokenizer...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model with 8-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_8bit=True,
    device_map="auto",
    trust_remote_code=True
)

# Prepare for LoRA
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n✅ Model loaded successfully!")

In [ ]:
# 데이터 로딩 및 전처리
# (실제로는 3rd Place의 데이터 처리 로직 사용)

# 예제: CSV를 Hugging Face Dataset으로 변환
import pandas as pd

def preprocess_data(df, tokenizer, max_length=512):
    """
    3rd Place 로직에 맞게 수정 필요
    """
    def tokenize_function(examples):
        # Qwen2.5 프롬프트 형식
        prompts = []
        for text in examples['text']:
            prompt = f"""<|im_start|>system
You are a medical expert tasked with extracting clinical entities.<|im_end|>
<|im_start|>user
{text}<|im_end|>
<|im_start|>assistant
"""
            prompts.append(prompt)
        
        return tokenizer(
            prompts,
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )
    
    # Dataset 변환
    from datasets import Dataset
    dataset = Dataset.from_pandas(df)
    tokenized = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    
    return tokenized

# 학습 데이터 로드
train_df = pd.read_csv('data/train_notes.csv')
print(f"Training samples: {len(train_df)}")

# 전처리
train_dataset = preprocess_data(train_df, tokenizer)

print("✅ Data prepared!")

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",  # Colab에서는 tensorboard 사용 가능
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

print("✅ Trainer initialized!")
print("\n⏰ 학습 시작! 예상 시간: 4-6시간")
print("💡 팁: Runtime → Manage sessions 에서 진행 상황 확인 가능")

In [ ]:
# 🚀 학습 시작!
trainer.train()

print("\n✅ Fine-tuning 완료!")
print(f"모델 저장 위치: {OUTPUT_DIR}")

## Step 8: 모델 저장 및 병합

In [ ]:
# LoRA weights 저장
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ LoRA adapter saved to: {OUTPUT_DIR}")

# (Optional) Base model과 병합하여 독립 실행 가능한 모델 생성
# merge = input("Base model과 병합할까요? (y/n): ")
# if merge.lower() == 'y':
#     merged_model = model.merge_and_unload()
#     merged_output = OUTPUT_DIR + "_merged"
#     merged_model.save_pretrained(merged_output)
#     tokenizer.save_pretrained(merged_output)
#     print(f"✅ Merged model saved to: {merged_output}")

## Step 9: 모델 테스트

In [ ]:
# 간단한 추론 테스트
test_text = "Patient presented with chest pain and shortness of breath."

prompt = f"""<|im_start|>system
You are a medical expert. Extract clinical entities from the text.<|im_end|>
<|im_start|>user
{test_text}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Test Input:")
print(test_text)
print("\nModel Output:")
print(result)

## Step 10: 모델 다운로드

In [ ]:
# Google Drive에 이미 저장되어 있지만, 로컬로 다운로드하고 싶다면:

# 압축
import shutil
archive_path = f"{OUTPUT_DIR}.tar.gz"
shutil.make_archive(OUTPUT_DIR, 'gztar', OUTPUT_DIR)

print(f"✅ 압축 완료: {archive_path}")
print(f"크기: {os.path.getsize(archive_path) / 1024 / 1024:.1f} MB")

# 다운로드 (크기가 작으면 가능)
# from google.colab import files
# files.download(archive_path)

## 🎉 완료!

다음 단계:
1. Google Drive에서 모델 다운로드
2. 로컬 환경에 복사
3. 추론 실행:

```bash
python main.py \
  --base_model_path Qwen/Qwen2.5-7B-Instruct \
  --model_path_peft path/to/downloaded/model
```

---

**추가 학습이 필요하면 이 노트북을 다시 실행하세요!**

- Classification 학습: `TASK = "classification"` 으로 변경
- 다른 청크 크기: `CHUNK_SIZE = 500` 으로 변경